In [ ]:
import pandas as pd
from bs4 import BeautifulSoup
import re
import os

# -------------------------------------------------------------
# 1. Import the Excel File
# -------------------------------------------------------------
excel_path = r"F:\Chimney Work\Marketing\Parivesh Work\New\Combined File_FULL.xlsx"

if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the Excel file at: {excel_path}")

df = pd.read_excel(excel_path)

if 'Project Details XML' not in df.columns:
    raise KeyError("The Excel file does not contain a column named 'Project Details XML'")

print(f"Successfully loaded {len(df)} rows. Commencing precise HTML/XML parsing...")

# -------------------------------------------------------------
# 2. Initialize Target Framework Columns with "N/A"
# -------------------------------------------------------------
# (Emails are kept as is since they were capturing fine previously)
for i in range(1, 2):
    df[f'Email_{i}'] = "N/A"

for i in range(1, 2):
    df[f'Mobile_{i}'] = "N/A"
    df[f'Landline_{i}'] = "N/A"

# Regex for standard email filtering (used as fallback/addition)
email_regex = re.compile(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}')

# -------------------------------------------------------------
# 3. HTML/XML DOM Structure Extraction Loop
# -------------------------------------------------------------
for idx, row in df.iterrows():
    xml_data = str(row['Project Details XML'])
    
    if pd.isna(row['Project Details XML']) or xml_data == "N/A" or xml_data.strip() == "":
        continue
        
    # Cook the soup using the HTML parser to navigate table structures
    soup = BeautifulSoup(xml_data, 'html.parser')
    
    # --- A. Precise Mobile & Landline Targeted Row Extraction ---
    mobiles = []
    landlines = []
    
    # Find all table rows across the document segment
    rows = soup.find_all('tr')
    
    for r in rows:
        cells = r.find_all(['td', 'th'])
        # We need at least a label cell and a value cell to extract meaningful data
        if len(cells) >= 2:
            # Clean and normalize the text from the label column
            label = cells[0].get_text(strip=True).lower()
            value = cells[1].get_text(strip=True)
            
            # Clean out common character noise from the phone strings
            clean_val = re.sub(r'[^\d,/-]', '', value).strip()
            
            # Split values in case multiple numbers are listed together (e.g., separated by commas or slashes)
            split_numbers = [num.strip() for num in re.split(r'[,/|-]', clean_val) if num.strip()]
            
            if "mobile" in label or "mo. no" in label:
                for num in split_numbers:
                    # Clean out noise and check standard length constraints
                    if len(num) >= 10 and num not in mobiles:
                        mobiles.append(num)
                        
            elif "landline" in label or "phone" in label or "tel. no" in label:
                for num in split_numbers:
                    if len(num) >= 6 and num not in landlines and num not in mobiles:
                        landlines.append(num)

    # --- B. Email Fallback Parsing ---
    found_emails = list(dict.fromkeys(email_regex.findall(xml_data)))
    found_emails = [e for e in found_emails if not e.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.pdf'))]
    
    # --- C. Distribute Data into Columns ---
    for i in range(min(2, len(found_emails))):
        df.at[idx, f'Email_{i+1}'] = found_emails[i]
        
    for i in range(min(3, len(mobiles))):
        df.at[idx, f'Mobile_{i+1}'] = mobiles[i]
        
    for i in range(min(3, len(landlines))):
        df.at[idx, f'Landline_{i+1}'] = landlines[i]

# -------------------------------------------------------------
# 4. Save the New Columns Back to the Original File
# -------------------------------------------------------------
print("Saving clean structural details back to your target Excel path...")
df.to_excel(excel_path, index=False)
print(f"🎉 Task complete! Clean elements targeted and saved at: {excel_path}")

Successfully loaded 472 rows. Commencing precise HTML/XML parsing...
Saving clean structural details back to your target Excel path...
🎉 Task complete! Clean elements targeted and saved at: F:\Chimney Work\Marketing\Parivesh Work\Leads 2025-26.xlsx
